In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import re
import torch
from multiprocessing import Pool

from tqdm import tqdm
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset as HFDataset, load_dataset, concatenate_datasets
from transformers import (
    RobertaTokenizer, 
    RobertaModel,
    RobertaForMaskedLM,
    BatchEncoding
)

from embed_stage2 import (
    prepare_token_dataset,
    train_model, 
    doc_sim_score_with_cl, 
    ContrastiveLearningDataset, 
    ContrastiveLearningModel
)

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c107WarningC1ENS_7variantIJNS0_11UserWarningENS0_18DeprecationWarningEEEERKNS_14SourceLocationENSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEEb'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and yo

In [3]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaModel.from_pretrained('./roberta-tuned-v1', add_pooling_layer=False, output_hidden_states=True)

In [4]:
BLOCK_SIZE = 128 # Stride length when splitting long texts into 512-length segments
MAX_SEQ_LEN = 512 # maximum length of a sequence that BERT can operate on

posting_txt_col = 'description'
resume_txt_col = 'Resume_str'

DEVICE = (f'cuda:0' if torch.cuda.is_available() else 'cpu')

In [5]:
# ------------------------------------------------------------
# Utility: tokenize text into sentences (simple heuristic)
# ------------------------------------------------------------
def split_sentences(text):
    # Keeps newlines intact but splits on ., ?, !
    text = text.replace("\n", " <N> ")  # temporary marker
    sents = re.split(r'(?<=[.!?])\s+', text)
    sents = [s.replace("<N>", "\n") for s in sents]
    return [s.strip() for s in sents if s.strip()]


# ============================================================
# 1. MASKED TOKEN AUGMENTATION
#    Randomly mask a percentage of tokens for MLM-type contrastive learning
# ============================================================
def random_mask(text, mask_prob=0.15):
    tokens = tokenizer.tokenize(text)
    new_tokens = []
    for t in tokens:
        if np.random.random() < mask_prob:
            new_tokens.append(tokenizer.mask_token)
        else:
            new_tokens.append(t)
    return tokenizer.convert_tokens_to_string(new_tokens)


# ============================================================
# 2. SENTENCE DROPOUT
#    Randomly drop 1–3 sentences from the document
# ============================================================
def sentence_dropout(text, drop_prob=0.25):
    sentences = split_sentences(text)
    keep = [s for s in sentences if np.random.random() > drop_prob]
    if not keep:  # Ensure at least one sentence
        keep = np.random.choice(sentences, 1)
    return " ".join(keep)


# ============================================================
# 3. SENTENCE SHUFFLING
#    Randomly shuffle sentences within the document
# ============================================================
def sentence_shuffle(text):
    sentences = split_sentences(text)
    np.random.shuffle(sentences)
    return " ".join(sentences)


# ============================================================
# 4. RANDOM SPAN DELETION
#    Delete one or more random spans of tokens
# ============================================================
def random_span_deletion(text, num_spans=1, span_length=3):
    tokens = tokenizer.tokenize(text)
    n = len(tokens)
    for _ in range(num_spans):
        if n <= span_length:
            break
        start = np.random.randint(0, max(0, n - span_length))
        del tokens[start:start+span_length]
        n = len(tokens)
    return tokenizer.convert_tokens_to_string(tokens)

In [6]:
lm_dataset = prepare_token_dataset(tokenizer, posting_path='./linkedin_data/sample_postings.csv', resume_path='./resume_data/sample_resumes.csv')
# lm_dataset = prepare_token_dataset(tokenizer, posting_path='./temp/postings5k.csv', resume_path='./resume_data/Resume.csv')


num_proc must be <= 10. Reducing num_proc to 10 for dataset of size 10.


Map (num_proc=12):   0%|          | 0/66 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/12 [00:00<?, ? examples/s]

In [7]:
augmentation_fns = [
        random_mask,
        sentence_dropout,
        sentence_shuffle,
        random_span_deletion,
    ]

In [8]:
# lm_dataset.set_format('torch')
d = ContrastiveLearningDataset(lm_dataset, 'train', tokenizer, augmentation_fns)

100%|██████████| 171/171 [00:01<00:00, 97.57it/s] 


In [11]:
# loader = DataLoader(d, batch_size=32, shuffle=False)
# m = ContrastiveLearningModel(model, 448).to(DEVICE)
# y = next(iter(loader))

In [ ]:
# y['input_ids'][:, 0].shape

In [ ]:
# m(y)[0].shape

In [ ]:
# m.get_embedding(BatchEncoding({'input_ids': y['input_ids'][:, 0], 'attention_mask': y['attention_mask'][:, 0]}).to(DEVICE))

In [ ]:
# model({'input_ids': y['input_ids'][:, 0], 'attention_mask': y['attention_mask'][:, 0]})

In [ ]:
# m = ContrastiveLearningModel(model)
# m(next(iter(loader)))

In [ ]:
# from tqdm import tqdm
# for i in tqdm(list(range(len(d)))):
#     x = d[i]

In [15]:
batch_size = 16
lr = 5e-5
m = ContrastiveLearningModel(model).to(DEVICE)
optimizer = optim.Adam(m.parameters(), lr=lr)
loss_fn = nn.TripletMarginLoss()
save_path = None#'./temp/contrastive_learning.pth'

epochs = 10

train_model(m, optimizer, d, loss_fn, epochs, batch_size, device=DEVICE, save_path=save_path, save_freq=1)


Learning rate: 5e-05
No learning rate scheduling!
Training for 10 epochs, with batch size=16
Using device: cuda:0

-----Epoch 1/10-----
Batch 11/11, loss: 0.3340327902273698 (0.548s)


100%|██████████| 171/171 [00:01<00:00, 106.72it/s]



-----Epoch 2/10-----
Batch 11/11, loss: 0.15963984551754865 (0.550s)


100%|██████████| 171/171 [00:01<00:00, 107.86it/s]



-----Epoch 3/10-----
Batch 11/11, loss: 0.09086628529158505 (0.548s)


...

100%|██████████| 171/171 [00:01<00:00, 109.39it/s]



-----Epoch 9/10-----
Batch 11/11, loss: 0.05079725994305177 (0.549s)


100%|██████████| 171/171 [00:01<00:00, 106.86it/s]



-----Epoch 10/10-----
Batch 11/11, loss: 0.027614139359105717 (0.549s)


100%|██████████| 171/171 [00:01<00:00, 108.30it/s]


In [ ]:
from embed_stage2 import fetch_all_postings_text, fetch_all_resumes_text, load_model

posting_db_url = 'sqlite:///postings.db'
resume_db_url = 'sqlite:///resumes.db'
postings = fetch_all_postings_text(posting_db_url)
resumes = fetch_all_resumes_text(resume_db_url)

mm = ContrastiveLearningModel(model).to(DEVICE)
mm = load_model(mm, save_path)

In [ ]:
t = doc_sim_score_with_cl(m, postings[2], resumes[4], DEVICE)
t.cpu().item()

In [ ]:
# loader = DataLoader(d, batch_size=32, shuffle=False)
# y = next(iter(loader))

# mm({k: v.to(DEVICE) for k,v in y.items()})